# 06 Baseline Comparison

Compute reconstruction scores for the trained convolutional autoencoder, select thresholds only on validation data, and evaluate the held-out test split with frozen thresholds.

In [5]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_curve
)

from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

In [2]:
# -----------------------------
# paths
# -----------------------------
REPO_ROOT = Path("../..").resolve()

DATA_ROOT = REPO_ROOT / "data/03_ma_dataset"
TEST_DIR = DATA_ROOT / "test"

IMAGE_SIZE = (150, 100)

# outputs
OUTPUT_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(exist_ok=True)

SCORES_PATH = OUTPUT_DIR / "ae_scores.csv"
THRESHOLD_PATH = OUTPUT_DIR / "ae_threshold.json"
METRICS_PATH = OUTPUT_DIR / "ae_metrics.csv"

val_df = pd.read_csv(OUTPUT_DIR / "val_scores.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test_scores.csv")

print(val_df.head())

       filename  label  global_mse  global_mae   ver_max  ver_topk
0  65qtta2q.png      0    0.000341    0.009769  0.000754  0.000732
1  trjumm0y.png      0    0.000316    0.009562  0.000739  0.000706
2  dq08sg40.png      0    0.000302    0.009258  0.000739  0.000698
3  c8vwg5zg.png      0    0.000301    0.009263  0.000735  0.000725
4  zq48enwb.png      0    0.000289    0.009458  0.000730  0.000644


In [3]:
def load_data(folder, csv_path, target_size=(150, 100)):
    df = pd.read_csv(csv_path)

    images, labels, filenames = [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        file_path = folder / row["filename"]

        
        if not file_path.exists():
            print(f"Missing: {file_path}")
            continue


        img = Image.open(file_path).convert("RGB").resize(target_size)
        img = np.array(img).astype("float32") / 255.0

        images.append(img)
        labels.append(row["label"])   # 0/1 bleibt
        filenames.append(row["filename"])

    return np.array(images), np.array(labels), np.array(filenames)

In [4]:
X_all, y_all, files_all = load_data(
    TEST_DIR,
    TEST_DIR / "labels.csv"
)

# Split: 50% val / 50% test
X_val, X_test, y_val, y_test, f_val, f_test = train_test_split(
    X_all, y_all, files_all,
    test_size=0.5,
    stratify=y_all,
    random_state=42
)

100%|██████████| 5000/5000 [05:10<00:00, 16.11it/s] 


In [6]:
# Flatten for classic ML models

X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

In [7]:
# Train only on normal data (label=0)
X_train_norm = X_val_flat[y_val == 0]

## Isolation Forest Baseline

In [8]:
iso = IsolationForest(
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train_norm)

,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


#### Calculate Isolation Forest scores for validation and test sets 

In [9]:
val_iso = -iso.decision_function(X_val_flat)
test_iso = -iso.decision_function(X_test_flat)

## PCA Baseline

In [10]:
pca = PCA(n_components=50, random_state=42)
pca.fit(X_train_norm)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",50
,"random_state random_state: int, RandomState instance or None, default=NoneUsed when the 'arpack' or 'randomized' solvers are used. Pass an intfor reproducible results across multiple function calls.See :term:`Glossary <random_state>`... versionadded:: 0.18.0",42
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to e

In [11]:
# Reconstruction error for PCA
def pca_reconstruction_error(pca, X):
    X_proj = pca.transform(X)
    X_recon = pca.inverse_transform(X_proj)
    return ((X - X_recon) ** 2).mean(axis=1)

# Scores

val_pca = pca_reconstruction_error(pca, X_val_flat)
test_pca = pca_reconstruction_error(pca, X_test_flat)

### Integrate Baselines to csv

In [12]:
val_df["iso_forest"] = val_iso
test_df["iso_forest"] = test_iso

val_df["pca_recon"] = val_pca
test_df["pca_recon"] = test_pca

In [13]:
print(val_df[["iso_forest", "pca_recon"]].describe())

        iso_forest    pca_recon
count  2500.000000  2500.000000
mean     -0.101886     0.000232
std       0.023578     0.000107
min      -0.138835     0.000125
25%      -0.112981     0.000221
50%      -0.105515     0.000230
75%      -0.097391     0.000237
max       0.113984     0.002675


### Save Dataframes with Baseline Scores

In [15]:
val_df.to_csv(OUTPUT_DIR / "val_scores_with_baselines.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_scores_with_baselines.csv", index=False)

# Evaluation Table